# Clase 021 — Aleatoriedad y semillas

**Parte 0** · NumPy `random.Generator`.

> 🎯 Aleatoriedad reproducible con el API moderno. Distribuciones, permutación, Monte Carlo.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import numpy as np
import math
rng = np.random.default_rng(seed=42)

## 1️⃣ API moderno vs legacy

```python
# ❌ legacy (deprecated)
np.random.seed(42)
np.random.normal(0, 1, 100)

# ✅ moderno (recomendado)
rng = np.random.default_rng(seed=42)
rng.normal(0, 1, 100)
```

**Ventajas del Generator**:
- Algoritmo más rápido (PCG64 vs Mersenne Twister)
- Múltiples generadores independientes (no estado global)
- API más limpio y consistente
- Mejor calidad estadística

## 2️⃣ Reproducibilidad

Mismo seed → exactamente los mismos números, siempre, en cualquier máquina:

In [ ]:
rng_a = np.random.default_rng(seed=42)
rng_b = np.random.default_rng(seed=42)

a = rng_a.normal(0, 1, 5)
b = rng_b.normal(0, 1, 5)
print(f'a: {a}')
print(f'b: {b}')
print(f'iguales? {np.array_equal(a, b)}')

## 3️⃣ Distribuciones más usadas

| Método | Distribución | Param típicos |
|---|---|---|
| `rng.random(n)` | Uniforme [0, 1) | — |
| `rng.uniform(lo, hi, n)` | Uniforme [lo, hi) | lo, hi |
| `rng.normal(μ, σ, n)` | Normal | media, std |
| `rng.standard_normal(n)` | Normal(0,1) | — |
| `rng.integers(lo, hi, n)` | Uniforme discreto [lo, hi) | lo, hi |
| `rng.binomial(n, p, size)` | Binomial | n trials, p éxito |
| `rng.poisson(λ, size)` | Poisson | tasa |
| `rng.exponential(scale, size)` | Exponencial | scale = 1/λ |
| `rng.gamma(shape, scale, size)` | Gamma | shape, scale |
| `rng.beta(a, b, size)` | Beta | a, b |

In [ ]:
N = 100_000
rng = np.random.default_rng(42)

# Comparar empírico vs teórico
muestras = {
    'uniform(0,1)'    : (rng.random(N),           0.5, math.sqrt(1/12)),
    'normal(5,2)'     : (rng.normal(5, 2, N),     5,   2),
    'exponential(3)'  : (rng.exponential(3, N),   3,   3),
    'poisson(4)'      : (rng.poisson(4, N),       4,   math.sqrt(4)),
}

print(f'{"distribución":20s} {"μ_emp":>8s} {"μ_teo":>8s} {"σ_emp":>8s} {"σ_teo":>8s}')
for nombre, (x, mu_t, sd_t) in muestras.items():
    print(f'{nombre:20s} {x.mean():8.3f} {mu_t:8.3f} {x.std():8.3f} {sd_t:8.3f}')

## 4️⃣ `choice` y `permutation`

```python
rng.choice(arr, size, replace=False, p=probabilidades)
rng.permutation(arr)   # mezcla, devuelve copia
rng.shuffle(arr)       # mezcla in-place
```

In [ ]:
rng = np.random.default_rng(42)
opciones = ['A', 'B', 'C', 'D']

# Muestreo con probabilidades distintas
muestra = rng.choice(opciones, size=20, p=[0.5, 0.3, 0.15, 0.05])
vals, cnts = np.unique(muestra, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'{v}: {c}')

# Permutación reproducible
rng2 = np.random.default_rng(42)
rng3 = np.random.default_rng(42)
print(f'\nperm1: {rng2.permutation([1,2,3,4,5])}')
print(f'perm2: {rng3.permutation([1,2,3,4,5])}  ← idéntica')

## 5️⃣ Monte Carlo de π

Lanzas puntos al azar en `[-1, 1] × [-1, 1]`. Razón "dentro del círculo unitario" / "total" tiende a `π/4`.

In [ ]:
for N in [1_000, 100_000, 10_000_000]:
    rng = np.random.default_rng(42)
    pts = rng.uniform(-1, 1, size=(N, 2))
    dentro = (pts[:, 0]**2 + pts[:, 1]**2 <= 1).sum()
    pi_est = 4 * dentro / N
    error = abs(pi_est - math.pi)
    print(f'N={N:>10,}  π≈{pi_est:.6f}  error={error:.6f}')

## 6️⃣ Bootstrap — distribución de un estadístico

Resampleas con reemplazo del sample original y recalculas el estadístico — obtienes una distribución del estimador sin asumir CLT.

In [ ]:
# Sample observado
rng = np.random.default_rng(42)
sample = rng.normal(50, 10, 30)
print(f'sample mean : {sample.mean():.2f}')

# Bootstrap: 10000 resamples
B = 10_000
bootstrap_means = np.array([
    rng.choice(sample, size=len(sample), replace=True).mean()
    for _ in range(B)
])

print(f'boot mean   : {bootstrap_means.mean():.2f}')
print(f'boot std    : {bootstrap_means.std():.2f}  (= SE de la media)')
print(f'95% CI boot : [{np.percentile(bootstrap_means, 2.5):.2f}, {np.percentile(bootstrap_means, 97.5):.2f}]')

## ✅ Checklist

- [ ] Uso `np.random.default_rng(seed)` no `np.random.seed`
- [ ] Sé generar normal, uniform, integers, binomial, poisson
- [ ] Mismo seed → mismo output reproducible
- [ ] Sé usar `choice` para muestreo con/sin reemplazo
- [ ] Implementé Monte Carlo y bootstrap

## 📝 Homework

Ver `README.md`. Monte Carlo de π, bootstrap CI, reproducibilidad, momentos empíricos vs teóricos.

## 🔗 Referencias

- [`random.Generator`](https://numpy.org/doc/stable/reference/random/generator.html)
- [NEP 19 RNG policy](https://numpy.org/neps/nep-0019-rng-policy.html)

➡️ **Siguiente:** [022 — Pandas: Series y DataFrame](../022-pandas-series-y-dataframe/README.md)